[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CityScope/TransitLOSStudies/blob/main/example_metro_area/example.ipynb)

# Transit level of service for any area

1. Choose an area: a place name and an area type (UN functional area, urban centre or municipality), a shape drawn on a map, or a file.
2. Choose the day, the hours and the transit feeds.
3. Run the study.
4. See the results on an interactive map and download a zip with the streets, the H3 hexagons and the census.

Every input comes from the most local source available: the area's own census and official transit feeds when there are, otherwise global data (WorldPop, Mobility Database, Transitland).

The level of service uses the standard scoring functions of the transitLOS package, the ones taken from the transit literature (stop score from the walking decay, headway, speed and mode). No curve is fitted to data here.

Run the cells in order. In Google Colab: **Runtime → Run all** also works with the defaults (Cambridge, MA).

## 1. Install

In [ ]:
# Gets the study code and sets everything up. The first run installs the study packages (a few minutes); nothing restarts.
import subprocess, sys
from pathlib import Path
if not Path("bootstrap.py").is_file():                 # not inside the study folder (Colab): use the clone, making it the first time
    folder = Path("TransitLOSStudies/example_metro_area")
    if not folder.is_dir():
        subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/CityScope/TransitLOSStudies.git"])
    sys.path.insert(0, str(folder.resolve()))
import bootstrap
study = bootstrap.setup()
import ui

## 2. Choose the area

Set the place and the area type. **Run all** uses these two values and nothing else, so it runs straight through.

`AREA` can be:
- `"metro_area"` (also `"functional_area"`): the UN functional urban area, the urban centre plus its commuting zone
- `"urban_core"`: the UN urban centre (the dense built-up city)
- `"municipality"`: the city's own boundary (the official one when the country publishes it)

To use your own area instead, set `AOI_FILE` (a `.gpkg`, `.geojson`, `.kml` or zipped shapefile; `"upload"` picks one from your computer in Colab) or `DRAW = True` (then draw the shape in section 3 and run the cells from there).

In [ ]:
PLACE = "Cambridge, Massachusetts, USA"
AREA = "municipality"

AOI_FILE = None       # a file with your own area, or "upload" (Colab)
DRAW = False          # True: draw the area on a map in section 3

## 3. The area

In [ ]:
if DRAW:                                            # only when you chose to draw: draw the shape, then run the cells below
    draw_map, drawn = ui.draw_area(study.area_from_place(PLACE, AREA))
    display(draw_map)

In [ ]:
if DRAW:
    assert drawn, "Nothing drawn: draw a polygon or rectangle on the map above, then run this cell and the ones below again."
    aoi = study.area_from_geojson(drawn[-1])
else:
    aoi = study.get_area(PLACE, AREA, ui.upload_file() if AOI_FILE == "upload" else AOI_FILE)
ui.show_area(aoi)

## 4. Choose the day and the hours

In [ ]:
DAY = "weekday"         # "weekday" (a working day), "saturday" or "sunday"
START_TIME = "06:00"
END_TIME = "22:00"

## 5. Choose the transit feeds

- `"download"`: find the area's feeds online: the country's official portal when there is one, plus the feeds it misses from the Mobility Database and Transitland.
  If a source needs an API key, the run asks for it and says where to get one. Press Enter to skip that source.
- A folder with your GTFS feeds, e.g. `"my_feeds/"`.
- A list of feeds: zip files, folders or download links.

In [ ]:
FEEDS = "download"
# FEEDS = "my_feeds/"
# FEEDS = ["bus_gtfs.zip", "https://example.com/rail_gtfs.zip"]
# FEEDS = ui.upload_feeds()      # Colab: choose GTFS zip files from your computer

## 6. Run

This gets the streets, the transit feeds and the census for the area, then computes everything.
A municipality takes from a few minutes to an hour; a whole metro area takes longer.

In [ ]:
zip_path = study.run(aoi, day=DAY, start_time=START_TIME, end_time=END_TIME, feeds=FEEDS)
print(zip_path)

## 7. See the results on the map

The map shows the finest H3 hexagons or circles and the finest census level, with the stops and routes. The chart button (bottom left) opens the distribution, regression, ANOVA, discretization and metadata tabs.

In [ ]:
ui.show_map(zip_path)

## 8. Download the results

The zip holds:
- `streets.parquet`: every street segment with its transit level of service
- `h3_res<N>.parquet`: H3 hexagons at each resolution, with population, level of service and census columns
- `census_<level>.parquet`: the census at its finest level, with the same columns

In [ ]:
ui.download(zip_path)